# A maturity payoff, in Python

**DefinedFlow · Payoff mechanics**

Implement a hypothetical participation rule, inspect its inputs, and compare maturity scenarios.

For readers with basic Python knowledge. Use Python 3.11+ and run the cells from top to bottom. Keep `participation_payoff.py` beside this notebook and run the notebook with this folder as its working directory. The calculations use only the Python standard library; a Jupyter-compatible editor supplies the notebook runtime.

You will calculate a gross maturity payment, round it once to two decimals, and check scenarios and invalid inputs.

1. Set the inputs.
2. Apply the payoff rule.
3. Round the final amount.
4. Try other scenarios.
5. Check an invalid input.
6. Try a short exercise.

## 1. Set the inputs

All figures are illustrative. Assume a contract returns principal at maturity plus 80% of any positive reference-level return, and that the issuer pays in full. There are no fees, taxes, dividends, caps, barriers, or early redemption terms in this example.

The assumed rule is

$$M = N\left(1 + p\max\left(\frac{S_T}{S_0} - 1,\;0\right)\right).$$

Here $N$ is principal in currency units; $S_0$ and $S_T$ are initial and final reference levels on the same scale; $p$ is the participation fraction. This specific rule is an authored teaching example, not the terms of an issued product.

Strings preserve the exact decimal inputs when converted to `Fraction`. Enter `"0.8"` for 80%, not `"80"`.

In [1]:
from fractions import Fraction

principal = Fraction("1000")     # currency units
initial = Fraction("100")        # initial index level
final = Fraction("120")          # final index level
participation = Fraction("0.8")  # 80%, entered as a fraction

## 2. Apply the payoff rule

Only a positive reference-level return adds to the assumed principal repayment. The input levels below imply a 20% increase; 80% participation adds 160 currency units to the 1,000 principal.

In [2]:
level_return = final / initial - 1
positive_return = max(level_return, 0)
payment = principal * (1 + participation * positive_return)

print(f"Underlying return: {level_return * 100}%")
print(f"Unrounded payment: {payment}")

Underlying return: 20%
Unrounded payment: 1160


## 3. Round the final amount

Round once, half up to two decimals. The result is the **gross maturity payment including principal**. It is not profit, an annualized return, or a price today. Contractual rounding conventions can differ.

In [3]:
rounded = payment * 100 + Fraction(1, 2)
cents = rounded.numerator // rounded.denominator
result = f"{cents // 100}.{cents % 100:02d}"

print(result)

1160.00


## 4. Try other scenarios

Import the checked function from the adjacent Python file. It applies the same formula and rounding while also validating inputs.

The downside and flat scenarios return 1,000 under the stated assumptions; the upside scenario returns 1,160. These conditional calculations do not predict which scenario will occur.

In [4]:
from participation_payoff import maturity_payment

for final_level in ("80", "100", "120"):
    amount = maturity_payment("1000", "100", final_level, "0.8")
    print(f"Final level {final_level:>3} → {amount}")

Final level  80 → 1000.00
Final level 100 → 1000.00
Final level 120 → 1160.00


## 5. Check an invalid input

The initial reference level must be positive. The function rejects zero rather than dividing by it.

The function accepts ordinary nonnegative decimal strings with up to six decimal places and values no greater than 1,000,000,000. Principal and initial level must be positive, and participation must be between 0 and 1. These are limits of this teaching function. Signs, spaces, exponents, NaN, infinity, and numeric objects are rejected.

In [5]:
try:
    maturity_payment("1000", "0", "120", "0.8")
except ValueError as error:
    print(error)

principal and initial must be positive


## 6. Try a short exercise

Keep principal at 1,000 and the initial level at 100. If the final level is 125 and participation is 60%, calculate the gross maturity payment before running the next cell.

The level return is 25%. Multiplying it by 60% gives a 15% addition to principal, so the expected payment is 1,150.00 currency units.

In [6]:
exercise_payment = maturity_payment("1000", "100", "125", "0.6")
assert exercise_payment == "1150.00"
print(exercise_payment)

1150.00


## Limits and next steps

This notebook calculates the stated maturity rule only. It does not model issuer default, early-sale value, discounting, probabilities, or product valuation. Principal repayment depends on the issuer meeting its obligations; real products can have different protection conditions and payoff terms. See the SEC/FINRA and SEC references below.

A common mistake is to read `1160.00` as profit: it includes the `1000.00` principal. The increment in this illustrative scenario is `160.00`, before any excluded fees or taxes.

For an extension, hold the final level at 120 and compare participation fractions `"0"`, `"0.5"`, and `"1"`. Explain each result using the formula before running it.

## Sources

- [SEC and FINRA: Structured Notes with Principal Protection — Note the Terms of Your Investment](https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-bulletins/structured) — participation terms, maturity, and issuer risk.
- [SEC: Investor Bulletin — Structured Notes](https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-bulletins-76) — payoff structures and product risks.

Sources checked on 2026-11-12. They support the financial context; the numerical terms here are hypothetical. This educational example is not an investment recommendation.